# **Regression: Support Vector Regressor (SVR)**

## **Justification of Preprocessing Strategy**

### **The Scale and Margin Sensitivity of SVR**
Support Vector Regression (SVR) operates on a completely different principle than traditional regression. Instead of trying to minimize the error of every single point, SVR defines a "tube" of tolerance (controlled by the parameter $\epsilon$) around the regression line. Any predictions that fall within this $\epsilon$-tube are considered correct and do not contribute to the loss function. 
Because SVR calculates geometric distances to construct this tube and the optimal hyperplane, features with larger numerical ranges would completely skew the margin calculations. Therefore, **Standardization (`StandardScaler`)** and **Normalization (`MinMaxScaler`)** are mandatory. We will test both to see which spatial distribution helps the algorithm converge faster and more accurately.

### **The Scalability Challenge: Why We Downsampled to 15,000 Rows**
Non-linear SVR relies on kernel tricks (like RBF) whose computational complexity scales between $O(n^2)$ and $O(n^3)$. While our Support Vector Classifier (SVC) managed to process 50,000 rows in a reasonable timeframe, **SVR is mathematically much denser**. 
In classification, SVC only calculates distances for the points near the decision boundary (the sparse support vectors), ignoring the rest. In regression, SVR must calculate the distance penalty for almost every single continuous point outside the $\epsilon$-tube. 
Empirical testing showed that running GridSearchCV and Optuna on 50,000 rows caused severe computational bottlenecks (exceeding 2.5 hours without convergence). To maintain experimental viability and allow the cross-validation loops to complete, we executed a **strategic down-sampling to 15,000 rows**. This ensures we can still evaluate the algorithm's predictive mechanics without locking up the hardware.

## **Experiment Design**

We defined a tournament of 6 experiments (2 Scalers × 3 Optimization Levels) to identify the most robust configuration, logging **both Train and Test metrics** explicitly:

* **Baseline**: Default SVR parameters (`C=1.0`, `kernel='rbf'`, `epsilon=0.1`) in both scaled spaces.
* **GridSearchCV**: A targeted cross-validated search exploring the trade-off between the regularization penalty (`C`) and the margin of tolerance (`epsilon`) for both Linear and RBF kernels.
* **Optuna Optimization**: Bayesian optimization utilizing `TPESampler` to dynamically explore a continuous logarithmic space for `C`, `epsilon`, and `gamma`, minimizing the validation RMSE on our 15k sample.

In [2]:
import pandas as pd
import numpy as np
import time
import mlflow
import optuna
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Regression_SVR")

# 2. Data Loading and Preparation
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")

categorical_cols = [
    'gender', 'ethnicity', 'smoking_status', 'education_level',
    'employment_status', 'age_groups', 'weight_status', 'income_level'
]

# Apply One-Hot Encoding
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Separate features and target 
# Drop classification targets 
X = df_final.drop(["diagnosed_diabetes", "diabetes_stage", "diabetes_risk_score"], axis=1)
y = df_final['diabetes_risk_score']

# 3. Sub-sampling to 15,000 rows due to SVR computational complexity O(n^2) - O(n^3)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=15000, random_state=42
)

num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns
SEED = 42

def log_regression_metrics(y_tr_true, y_tr_pred, y_te_true, y_te_pred, duration):
    # Logs Train and Test metrics explicitly to monitor the Overfitting Gap
    # Train Partition Metrics
    mlflow.log_metric("rmse_train", mean_squared_error(y_tr_true, y_tr_pred) ** 0.5)
    mlflow.log_metric("mae_train", mean_absolute_error(y_tr_true, y_tr_pred))
    mlflow.log_metric("r2_train", r2_score(y_tr_true, y_tr_pred))
    
    # Test Partition Metrics
    mlflow.log_metric("rmse_test", mean_squared_error(y_te_true, y_te_pred) ** 0.5)
    mlflow.log_metric("mae_test", mean_absolute_error(y_te_true, y_te_pred))
    mlflow.log_metric("r2_test", r2_score(y_te_true, y_te_pred))
    
    mlflow.log_metric("fit_time", duration)

# ---------------------------------------------------------
# 6 RUNS (2 Scalers x 3 Optimization Levels)
# ---------------------------------------------------------
scalers = {
    "Standardization": StandardScaler(),
    "Normalization": MinMaxScaler()
}

for s_name, scaler in scalers.items():
    
    # Apply Feature Scaling
    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()
    X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

    # --- RUN 1 & 4: BASELINE ---
    with mlflow.start_run(run_name=f"SVR_{s_name}_Baseline_50k"):
        # Using cache_size=2000 to drastically speed up kernel computations
        model = SVR(kernel='rbf', C=1.0, epsilon=0.1, cache_size=2000)
        
        start_time = time.time()
        model.fit(X_train_scaled, y_train)
        duration = time.time() - start_time
        
        # Explicit Predictions
        y_pred_train_base = model.predict(X_train_scaled)
        y_pred_test_base = model.predict(X_test_scaled)
        
        mlflow.log_params(model.get_params())
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("optimization", "none_default")
        mlflow.log_param("row_count", 50000)
        
        log_regression_metrics(y_train, y_pred_train_base, y_test, y_pred_test_base, duration)

    # --- RUN 2 & 5: GRIDSEARCHCV ---
    with mlflow.start_run(run_name=f"SVR_{s_name}_GridSearch_50k"):
        # Compact grid to prevent exponential runtimes
        param_grid = [
            {'kernel': ['linear'], 'C': [1, 10], 'epsilon': [0.1, 1.0]},
            {'kernel': ['rbf'], 'C': [1, 10], 'epsilon': [0.1, 1.0], 'gamma': ['scale']}
        ]
        
        grid = GridSearchCV(
            SVR(cache_size=2000), 
            param_grid, 
            cv=KFold(n_splits=3, shuffle=True, random_state=SEED), # Reduced CV to 3-folds for time
            scoring='neg_root_mean_squared_error', 
            n_jobs=2 # Using n_jobs=2 to avoid aggressive RAM exhaustion
        )
        
        start_time = time.time()
        grid.fit(X_train_scaled, y_train)
        duration = time.time() - start_time
        
        best_svr_grid = grid.best_estimator_
        
        # Explicit Predictions
        y_pred_train_grid = best_svr_grid.predict(X_train_scaled)
        y_pred_test_grid = best_svr_grid.predict(X_test_scaled)
        
        mlflow.log_params(grid.best_params_)
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("optimization", "GridSearchCV")
        
        log_regression_metrics(y_train, y_pred_train_grid, y_test, y_pred_test_grid, duration)

    # --- RUN 3 & 6: OPTUNA ---
    def objective(trial):
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf'])
        c_val = trial.suggest_float('C', 0.1, 100, log=True)
        epsilon_val = trial.suggest_float('epsilon', 0.01, 2.0, log=True)
        
        if kernel == 'rbf':
            gamma_val = trial.suggest_float('gamma', 1e-3, 1.0, log=True)
            model_opt = SVR(kernel=kernel, C=c_val, epsilon=epsilon_val, gamma=gamma_val, cache_size=2000)
        else:
            model_opt = SVR(kernel=kernel, C=c_val, epsilon=epsilon_val, cache_size=2000)
        
        # Internal CV restricted to train subset
        scores = cross_val_score(
            model_opt, X_train_scaled, y_train, 
            cv=KFold(n_splits=3, shuffle=True, random_state=SEED), 
            scoring='neg_root_mean_squared_error', 
            n_jobs=2
        )
        return -scores.mean()

    with mlflow.start_run(run_name=f"SVR_{s_name}_Optuna_50k"):
        study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
        
        start_time = time.time()
        study.optimize(objective, n_trials=10) # 10 trials is a safe budget for SVR on 50k
        duration = time.time() - start_time
        
        # Retrain Best Found Model
        best_svr_optuna = SVR(**study.best_params, cache_size=2000)
        best_svr_optuna.fit(X_train_scaled, y_train)
        
        # Explicit Predictions
        y_pred_train_opt = best_svr_optuna.predict(X_train_scaled)
        y_pred_test_opt = best_svr_optuna.predict(X_test_scaled)
        
        mlflow.log_params(study.best_params)
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("optimization", "optuna")
        
        log_regression_metrics(y_train, y_pred_train_opt, y_test, y_pred_test_opt, duration)

[I 2026-05-22 20:39:30,309] A new study created in memory with name: no-name-ac22bbd9-42b4-4a64-b0f1-2dab6e9b59c4
[I 2026-05-22 20:39:38,666] Trial 0 finished with value: 0.4315096074486888 and parameters: {'kernel': 'rbf', 'C': 15.702970884055382, 'epsilon': 0.23852347578447078, 'gamma': 0.0029380279387035343}. Best is trial 0 with value: 0.4315096074486888.
[I 2026-05-22 20:52:03,601] Trial 1 finished with value: 0.7746128175980965 and parameters: {'kernel': 'linear', 'C': 39.67605077052987, 'epsilon': 0.24164826029897515}. Best is trial 0 with value: 0.4315096074486888.
[I 2026-05-22 20:54:44,423] Trial 2 finished with value: 0.7076965620753914 and parameters: {'kernel': 'linear', 'C': 81.23245085588688, 'epsilon': 0.8231433730995551}. Best is trial 0 with value: 0.4315096074486888.
[I 2026-05-22 20:55:14,815] Trial 3 finished with value: 0.8089263701480532 and parameters: {'kernel': 'linear', 'C': 0.35498788321965025, 'epsilon': 0.05012686302434878}. Best is trial 0 with value: 0.4

## Winner Run Selection (Priority Elimination Framework)

### Policy
A run is only eligible to win if it does NOT show evidence of overfitting or underfitting. Before applying the MAE/RMSE/R² decision rules, we require the Train→Test gaps to remain small enough to indicate acceptable generalization. Runs that memorize the training set or show a large Train/Test gap are disqualified regardless of metric rank.

### Selection Criteria (priority order)
1. **Generalization filter (mandatory):** runs with overfitting or underfitting are removed from consideration.
2. **Priority 1 (60%): Lowest MAE (Test)** — primary objective for regression accuracy.
3. **Priority 2 (30%): Lowest RMSE (Test)** — used to reject runs where RMSE grows disproportionately relative to MAE.
4. **Priority 3 (10%): Acceptable R² (Test)** — confirms explanatory quality.
5. **Tiebreaker: Lowest Fit Time** — if MAE, RMSE, and R² are effectively tied.

### Runs Summary

| Run | Scaler | MAE (Train) | MAE (Test) | RMSE (Train) | RMSE (Test) | R² (Train) | R² (Test) | Fit Time |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| SVR_Standardization_Baseline_50k | Standardization | 0.49974 | 0.65004 | 0.96498 | 1.11619 | 0.98843 | 0.98488 | 11.81s |
| SVR_Standardization_GridSearch_50k | Standardization | 0.09268 | 0.34554 | 0.11920 | 0.58030 | 0.99982 | 0.99591 | 557.19s |
| SVR_Standardization_Optuna_50k | Standardization | 0.21797 | 0.23125 | 0.38369 | 0.40905 | 0.99817 | 0.99797 | 1314.80s |
| SVR_Normalization_Baseline_50k | Normalization | 0.45060 | 0.55513 | 0.77118 | 0.87559 | 0.99261 | 0.99069 | 9.95s |
| SVR_Normalization_GridSearch_50k | Normalization | 0.15479 | 0.36539 | 0.42767 | 0.64757 | 0.99773 | 0.99491 | 180.86s |
| SVR_Normalization_Optuna_50k | Normalization | 0.39896 | 0.40051 | 0.70541 | 0.70473 | 0.99382 | 0.99397 | 285.63s |

### Generalization Check (Test − Train)
- **SVR_Standardization_Baseline_50k:** MAE gap = 0.65004 − 0.49974 = **+0.15030** and RMSE gap = 1.11619 − 0.96498 = **+0.15121** → PASS.
- **SVR_Standardization_GridSearch_50k:** MAE gap = 0.34554 − 0.09268 = **+0.25286** and RMSE gap = 0.58030 − 0.11920 = **+0.46110** → FAIL (overfitting).
- **SVR_Standardization_Optuna_50k:** MAE gap = 0.23125 − 0.21797 = **+0.01328** and RMSE gap = 0.40905 − 0.38369 = **+0.02536** → PASS.
- **SVR_Normalization_Baseline_50k:** MAE gap = 0.55513 − 0.45060 = **+0.10453** and RMSE gap = 0.87559 − 0.77118 = **+0.10442** → PASS.
- **SVR_Normalization_GridSearch_50k:** MAE gap = 0.36539 − 0.15479 = **+0.21060** and RMSE gap = 0.64757 − 0.42767 = **+0.21990** → FAIL (overfitting).
- **SVR_Normalization_Optuna_50k:** MAE gap = 0.40051 − 0.39896 = **+0.00155** and RMSE gap = 0.70473 − 0.70541 = **−0.00068** → PASS.

### Overfitting / Underfitting Validation
- The GridSearch runs are disqualified because they memorize the training set too aggressively, producing large Train/Test gaps on MAE and RMSE.
- None of the remaining runs shows underfitting. Their Test R² values remain high, and the errors are still low.
- The Optuna and baseline runs that remain eligible have stable generalization behavior.

### Step-by-Step Elimination
**Step 1 — Apply the generalization filter**
- Passing runs: SVR_Standardization_Baseline_50k, SVR_Standardization_Optuna_50k, SVR_Normalization_Baseline_50k, SVR_Normalization_Optuna_50k.
- Disqualified runs: SVR_Standardization_GridSearch_50k, SVR_Normalization_GridSearch_50k.

**Step 2 — Compare Test MAE (Priority 1 — 60%)**
- SVR_Standardization_Optuna_50k: 0.23125
- SVR_Normalization_Optuna_50k: 0.40051
- SVR_Normalization_Baseline_50k: 0.55513
- SVR_Standardization_Baseline_50k: 0.65004
- Lowest MAE: **SVR_Standardization_Optuna_50k**.

**Step 3 — Compare Test RMSE (Priority 2 — 30%)**
- SVR_Standardization_Optuna_50k: 0.40905
- SVR_Normalization_Optuna_50k: 0.70473
- SVR_Normalization_Baseline_50k: 0.87559
- SVR_Standardization_Baseline_50k: 1.11619
- SVR_Standardization_Optuna_50k remains the best choice.

**Step 4 — Check Test R² (Priority 3 — 10%)**
- SVR_Standardization_Optuna_50k: 0.99797
- SVR_Normalization_Optuna_50k: 0.99397
- SVR_Normalization_Baseline_50k: 0.99069
- SVR_Standardization_Baseline_50k: 0.98488
- SVR_Standardization_Optuna_50k also leads on R².

### Final Decision
**Winner: SVR_Standardization_Optuna_50k**

**Justification:** `SVR_Standardization_Optuna_50k` is the strongest run among those that pass the generalization filter. It has the lowest Test MAE, the lowest Test RMSE, and the highest Test R². Fit time is not needed as a tiebreaker.

## Winner Hyperparameters
| Parameter | Value |
|---|---|
| **kernel** | rbf |
| **C** | 15.702970884055382 |
| **epsilon** | 0.23852347578447078 |
| **gamma** | 0.0029380279387035343 |
| **cache_size** | 2000 |

## Overfitting / Underfitting Diagnosis
- The GridSearch runs are disqualified because their Train/Test gaps are too large, especially on MAE and RMSE.
- The Optuna standardization run generalizes best and is the most balanced choice.
- The remaining eligible runs do not show underfitting.